In [1]:
# ==============================================================================
# Step 19: Load Cleaned Data
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# Clean columns again if starting fresh in this notebook
train_df.columns = train_df.columns.str.lower().str.strip().str.replace(' ', '_')
test_df.columns = test_df.columns.str.lower().str.strip().str.replace(' ', '_')

TARGET = 'y'
test_id = test_df['id'] # Save for submission
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

print("Step 19 Completed: Cleaned data loaded successfully.")
print("Key Finding: Datasets are ready for mathematical transformations.\n")


Step 19 Completed: Cleaned data loaded successfully.
Key Finding: Datasets are ready for mathematical transformations.



In [2]:
# ==============================================================================
# Step 20: Feature Engineering
# ==============================================================================
def engineer_features(df):
    df_eng = df.copy()
    
    # Feature 1: Previous Contact Status
    df_eng["was_previously_contacted"] = np.where(df_eng["pdays"] == -1, 0, 1)
    
    # Feature 2: Balance Status
    df_eng["balance_status"] = np.where(df_eng["balance"] >= 0, "non_negative", "negative")
    
    # Feature 3: Campaign Intensity
    df_eng["campaign_intensity"] = pd.cut(
        df_eng["campaign"],
        bins=[-np.inf, 1, 3, np.inf],
        labels=["low", "medium", "high"]
    ).astype("object")
    
    return df_eng

train_engineered = engineer_features(train_df)
test_engineered = engineer_features(test_df)

print("Step 20 Completed: Feature Engineering applied.")
print("--- Newly Added Columns ---")
display(train_engineered[['was_previously_contacted', 'balance_status', 'campaign_intensity']].head())
print("Key Finding: Categorizing continuous data helps tree-based models capture non-linear behaviors faster.\n")


Step 20 Completed: Feature Engineering applied.
--- Newly Added Columns ---


,was_previously_contacted,balance_status,campaign_intensity
0,0,non_negative,medium
1,0,non_negative,low
2,0,non_negative,medium
3,0,non_negative,medium
4,0,non_negative,low


Key Finding: Categorizing continuous data helps tree-based models capture non-linear behaviors faster.



In [3]:
# ==============================================================================
# Step 21: Prepare Features and Target
# ==============================================================================
X_train = train_engineered.drop(columns=[TARGET])
y_train = train_engineered[TARGET]
X_test = test_engineered.copy()

print("Step 21 Completed: Features and target successfully separated.")
print(f"Key Finding: X_train shape: {X_train.shape}, y_train shape: {y_train.shape}.\n")


Step 21 Completed: Features and target successfully separated.
Key Finding: X_train shape: (750000, 19), y_train shape: (750000,).



In [4]:
# ==============================================================================
# Step 22: Build Preprocessing Pipeline
# ==============================================================================
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

print("Step 22 Completed: Preprocessing pipeline built.")
print("Key Finding: Pipeline handles missing value imputation, scaling, and one-hot encoding seamlessly to prevent data leakage.\n")


Step 22 Completed: Preprocessing pipeline built.
Key Finding: Pipeline handles missing value imputation, scaling, and one-hot encoding seamlessly to prevent data leakage.



In [5]:
# ==============================================================================
# Step 23: Transform Datasets & Save Preprocessed Data
# ==============================================================================
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.joblib')

np.save('../data/X_train_processed.npy', X_train_processed)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/X_test_processed.npy', X_test_processed)
np.save('../data/test_id.npy', test_id.values)

print("Step 23 Completed: Datasets transformed and saved locally.")
print(f"Key Finding: Final feature count expanded from {X_train.shape[1]} to {X_train_processed.shape[1]} after One-Hot Encoding.\n")


Step 23 Completed: Datasets transformed and saved locally.
Key Finding: Final feature count expanded from 19 to 57 after One-Hot Encoding.

